# Instructor Notebook  --  Operation GRIDLOCK

---

| Phase | Time | Action |
|-------|------|--------|
| Reveal budget + show all designs | 10 min | Section 1 |
| Blue Team quick adjustment | 10 min | Students re-run BT cell |
| Red Team attack planning | 25 min | Students work in RT notebook |
| Live simulation (one matchup at a time) | 40 min | Section 2 |
| Debrief | 30 min | Section 3 |


In [ ]:
from gridlock_core import *
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import glob, os
print('Notebook ready.')

---
## Section 1 -- Reveal Budget & Show Designs

In [ ]:
RED_BUDGET_REVEALED = 80   # <- set this, then project

print("=" * 56)
print("  OPERATION GRIDLOCK  --  ATTACK BUDGET REVEALED")
print()
print("  Red Team budget:  " + str(RED_BUDGET_REVEALED) + " units")
print("  Range shown to Blue Teams:  " + str(RED_MIN) + "u -- " + str(RED_MAX) + "u")
print()
print("  Attack costs (flat):")
print("  Action | Cost")
print("  sever:   " + str(ATK_SEVER) + "u  (flat -- any cap)")
print("  degrade: " + str(ATK_DEGRADE) + "u  (flat -- any cap)")
print()
print("  Any unprotected link can be attacked.")
print("  Blue Teams: 10 min quick adjustment.")
print("  Red Teams:  25 min attack planning.")
print("=" * 56)


In [ ]:
NETWORK_FILES = sorted(glob.glob("BT*_network.csv"))
print("Found", len(NETWORK_FILES), "Blue Team files")
n = len(NETWORK_FILES)
if n == 0:
    print("No files found -- place BT*_network.csv here.")
else:
    cols_n = min(n, 3)
    rows_n = (n + cols_n - 1) // cols_n
    fig, axes = plt.subplots(rows_n, cols_n,
                              figsize=(11*cols_n, 10*rows_n),
                              facecolor="white")
    ax_list = list(axes.flat) if hasattr(axes, "flat") else [axes]
    for i, fpath in enumerate(NETWORK_FILES):
        nd, ed, mt = load_network_csv(fpath)
        nc = len(nd) * RELAY_NODE_COST
        ec = sum(link_build_cost(r.source, r.target, int(r.cap), bool(r.armored))
                 for r in ed.itertuples())
        n_arm = ed["armored"].sum()
        title = (mt.get("team_id","?") + " -- " + mt.get("team_name","?")
                 + "  |  " + str(nc+ec) + "/" + str(BLUE_BUDGET) + "u"
                 + "  " + str(n_arm) + " armored  " + mt.get("strategy","?"))
        draw_network(nd, ed, ax=ax_list[i], title=title, show_candidates=False)
    for j in range(n, rows_n*cols_n):
        ax_list[j].set_visible(False)
    plt.suptitle("OPERATION GRIDLOCK -- All Blue Team Designs",
                 color=PAL["text"], fontsize=15, fontweight="bold", y=1.01)
    plt.tight_layout()
    plt.savefig("all_designs.png", dpi=120,
                bbox_inches="tight", facecolor="white")
    plt.show()
    print("Saved -> all_designs.png")


---
## Section 2 -- Live Matchups

Change `SIM_NETWORK` and `SIM_ATTACK` for each pair.

In [ ]:
SIM_NETWORK = "BT1_network.csv"
SIM_ATTACK  = "RT1_vs_BT1_attacks.csv"

nodes_df, edges_df, net_meta = load_network_csv(SIM_NETWORK)
atk_df,   atk_meta           = load_attack_csv(SIM_ATTACK)
BT_ID = net_meta.get("team_id","?")
RT_ID = atk_meta.get("red_team_id","?")
PRED  = float(atk_meta.get("predicted_min","0"))

n_arm = edges_df["armored"].sum()
n_unp = len(edges_df) - n_arm
print("=" * 58)
print("  " + RT_ID + "  ->  " + BT_ID + " -- " + net_meta.get("team_name","?"))
print("  Strategy:  " + net_meta.get("strategy","?"))
print("  Plan:      " + atk_meta.get("chosen_plan","?"))
print("  Links:     " + str(len(edges_df))
      + "  (" + str(n_unp) + " unprotected, " + str(n_arm) + " armored)")
print("  RT prediction: min = " + str(round(PRED*100)) + "%")
print("=" * 58)

curve, events, dead_e, deg_e = run_attacks(nodes_df, edges_df, atk_df)
sc = score_resilience(curve)
print()
total_spent = 0
for ev in events:
    total_spent += ev["cost"]
    print("  Step " + str(ev["step"]) + ": " + ev["summary"])
    iso = ("  Isolated: " + str(ev["isolated"])) if ev["isolated"] else ""
    print("    -> " + str(round(ev["frac"]*100)) + "%" + iso)
print()
print("  Budget spent: " + str(total_spent) + "u")
print("  Score: R1=" + str(sc["R1"])
      + "  R2=" + str(sc["R2"])
      + "  Total=" + str(sc["total"]) + "/100")


In [ ]:
fig = plt.figure(figsize=(20,10), facecolor="white")
gs  = gridspec.GridSpec(1, 2, figure=fig, wspace=0.05,
                         left=0.02, right=0.98, top=0.92, bottom=0.08)
ax_net   = fig.add_subplot(gs[0])
ax_curve = fig.add_subplot(gs[1])
draw_network(nodes_df, edges_df, ax=ax_net,
             title=BT_ID + " -- post-attack state",
             dead_edges=dead_e, degraded_edges=deg_e,
             show_candidates=False)
draw_curve(curve, ax_curve, label="Resilience -- " + BT_ID, pred=PRED)
plt.suptitle("OPERATION GRIDLOCK  |  " + RT_ID + "  ->  " + BT_ID,
             color=PAL["text"], fontsize=15, fontweight="bold")
out_f = BT_ID + "_vs_" + RT_ID + "_result.png"
plt.savefig(out_f, dpi=130, bbox_inches="tight", facecolor="white")
plt.show()
print("Saved ->", out_f)


---
## Section 3 -- Debrief

In [ ]:
ALL_SCORES = {}
for atk_path in sorted(glob.glob("RT*_vs_BT*_attacks.csv")):
    atk_df, atk_meta = load_attack_csv(atk_path)
    bt_id    = atk_meta.get("target_team_id","?")
    net_path = bt_id + "_network.csv"
    if not os.path.exists(net_path):
        print("WARNING: missing", net_path); continue
    nd, ed, nm = load_network_csv(net_path)
    curve, *_  = run_attacks(nd, ed, atk_df)
    sc         = score_resilience(curve)
    ALL_SCORES[bt_id] = {"team_name": nm.get("team_name","?"),
                          "strategy":  nm.get("strategy","?"),
                          "rationale": nm.get("rationale",""),
                          "curve": curve, **sc}
    print(bt_id.ljust(6) + nm.get("team_name","?").ljust(24)
          + " R1=" + str(sc["R1"]).rjust(5)
          + "  R2=" + str(sc["R2"]).rjust(5)
          + "  Score=" + str(sc["total"]).rjust(6))


### Leaderboard

In [ ]:
if ALL_SCORES:
    teams = sorted(ALL_SCORES, key=lambda t: -ALL_SCORES[t]["total"])
    x, w  = np.arange(len(teams)), 0.35
    fig, ax = plt.subplots(figsize=(12,5), facecolor="white")
    ax.set_facecolor(PAL["panel"])
    for sp in ax.spines.values(): sp.set_color(PAL["border"]); sp.set_linewidth(0.8)
    ax.tick_params(colors=PAL["muted"])
    b1 = ax.bar(x-w/2, [ALL_SCORES[t]["R1"] for t in teams], w,
                label="R1 Absorption (60%)", color="#339af0", alpha=0.88,
                edgecolor="white", linewidth=0.8)
    b2 = ax.bar(x+w/2, [ALL_SCORES[t]["R2"] for t in teams], w,
                label="R2 Floor (40%)", color="#f76707", alpha=0.88,
                edgecolor="white", linewidth=0.8)
    for bars in [b1,b2]:
        for bar in bars:
            h = bar.get_height()
            ax.text(bar.get_x()+bar.get_width()/2, h+0.5, str(round(h)),
                    ha="center", va="bottom",
                    color=PAL["text"], fontsize=9, fontweight="bold")
    ax.set_ylim(0,115); ax.set_xticks(x)
    ax.set_xticklabels([t+"\n"+ALL_SCORES[t]["team_name"] for t in teams],
                       color=PAL["text"], fontsize=9)
    ax.set_ylabel("Score (0-100)", color=PAL["muted"])
    ax.axhline(80, color=PAL["muted"], lw=0.8, ls="--", alpha=0.5)
    ax.set_title("OPERATION GRIDLOCK -- Leaderboard",
                 color=PAL["text"], fontsize=14,
                 fontweight="bold", fontfamily="monospace")
    ax.legend(framealpha=0.9, facecolor="white",
              edgecolor=PAL["border"], labelcolor=PAL["text"], fontsize=10)
    plt.tight_layout()
    plt.savefig("leaderboard.png", dpi=130, bbox_inches="tight", facecolor="white")
    plt.show()


### All Resilience Curves

In [ ]:
if ALL_SCORES:
    col_cycle = ["#1971c2","#2b8a3e","#e67700","#862e9c","#c92a2a","#0b7285"]
    teams = sorted(ALL_SCORES, key=lambda t: -ALL_SCORES[t]["total"])
    maxl  = max(len(ALL_SCORES[t]["curve"]) for t in teams)
    fig, ax = plt.subplots(figsize=(13,6), facecolor="white")
    ax.set_facecolor(PAL["panel"])
    for sp in ax.spines.values(): sp.set_color(PAL["border"]); sp.set_linewidth(0.8)
    ax.tick_params(colors=PAL["muted"])
    ax.axhline(0.80, ls="--", color=PAL["warn"], lw=1.0, alpha=0.7)
    ax.text(0.01, 0.82, "80% target",
            transform=ax.get_yaxis_transform(), color=PAL["warn"], fontsize=8)
    import matplotlib.ticker as _tk2
    ax.yaxis.set_major_formatter(
        _tk2.FuncFormatter(lambda v,_: str(round(v*100))+"%"))
    for i, tid in enumerate(teams):
        sc  = ALL_SCORES[tid]; c = sc["curve"]; col = col_cycle[i%len(col_cycle)]
        xv  = list(range(len(c)))
        ax.plot(xv, c, color=col, lw=2.0, marker="o", markersize=5,
                markerfacecolor="white", markeredgecolor=col, markeredgewidth=1.5,
                label=tid+" -- "+sc["team_name"]+"  ("+str(sc["total"])+"pts)")
        ax.annotate(tid, (xv[-1], c[-1]),
                    textcoords="offset points", xytext=(4,0),
                    color=col, fontsize=8, fontweight="bold")
    ax.set_ylim(-0.06,1.15); ax.set_xlim(-0.3, maxl-0.5)
    ax.set_xlabel("Attack step", color=PAL["muted"])
    ax.set_ylabel("Demand fraction served", color=PAL["muted"])
    ax.set_title("All Resilience Curves",
                 color=PAL["text"], fontsize=12,
                 fontweight="bold", fontfamily="monospace")
    ax.legend(framealpha=0.92, facecolor="white",
              edgecolor=PAL["border"], labelcolor=PAL["text"], fontsize=9)
    plt.tight_layout()
    plt.savefig("all_curves.png", dpi=130, bbox_inches="tight", facecolor="white")
    plt.show()


### Design Rationale Reveal

In [ ]:
for f in sorted(glob.glob("BT*_network.csv")):
    nd, ed, mt = load_network_csv(f)
    tid = mt.get("team_id","?")
    sc  = ALL_SCORES.get(tid,{})
    n_arm = ed["armored"].sum()
    nc    = len(nd)*RELAY_NODE_COST
    ec    = sum(link_build_cost(r.source,r.target,int(r.cap),bool(r.armored))
                for r in ed.itertuples())
    print("=" * 60)
    print("  " + tid + " -- " + mt.get("team_name","?"))
    print("  Strategy:  " + mt.get("strategy","?"))
    print("  Budget:    " + str(nc+ec) + "/" + str(BLUE_BUDGET) + "u")
    print("  Armored:   " + str(n_arm) + " / " + str(len(ed)) + " links")
    if sc:
        print("  Score:     R1=" + str(sc.get("R1","?"))
              + "  R2=" + str(sc.get("R2","?"))
              + "  Total=" + str(sc.get("total","?")) + "/100")
    print()
    print("  " + mt.get("rationale","(not provided)"))
    print()
